# Laboratorio Evaluado: WBCD y Evaluación Multimodelo

**Reto Autónomo: Clínica Oncológica**

Eres el Consultor IA de un hospital. El dataset WBCD contiene radios y perímetros de células. Debes entrenar k-NN y Árboles de Decisión, y validar qué pasa si no escalas los datos.

## 1. Carga de Datos y Exploración

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

# TODO: Carga el dataset WBCD desde UCI
# df_wbcd = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data", header=None)
# Agrega nombres de columnas si es necesario
base = ["radio", "textura", "perimetro", "area", "suavidad",
        "compacidad", "concavidad", "puntos_concavos", "simetria", "dimension_fractal"]
columnas = ["id", "diagnostico"] + [b + "_media" for b in base] + \
           [b + "_error" for b in base] + [b + "_peor" for b in base]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
df_wbcd = pd.read_csv(url, header=None, names=columnas)

print(df_wbcd.shape)
print(df_wbcd["diagnostico"].value_counts())
df_wbcd.head()

## 2. Preprocesamiento

In [ ]:
# TODO: Convierte la columna de diagnosis (M/B) a binario (1/0)
df_wbcd["diagnostico"] = df_wbcd["diagnostico"].map({"M": 1, "B": 0})

# TODO: Separa en X (features) e y (target)
X = df_wbcd.drop(columns=["id", "diagnostico"])
y = df_wbcd["diagnostico"]
# TODO: Haz el train_test_split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

## 3. Escalamiento (Solo para k-NN)

In [ ]:
# TODO: Aplica StandardScaler a X_train y X_test.
scaler = StandardScaler()
# Recuerda: .fit_transform() en train, y solo .transform() en test.
X_train_esc = scaler.fit_transform(X_train)

X_test_esc = scaler.transform(X_test)

## 4. Modelado y Matrices de Confusión

In [ ]:
# TODO: Entrena un KNeighborsClassifier(n_neighbors=5)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_esc, y_train)
pred_knn = knn.predict(X_test_esc)
# k-NN SIN escalar
knn_sin = KNeighborsClassifier(n_neighbors=5)
knn_sin.fit(X_train, y_train)
pred_knn_sin = knn_sin.predict(X_test)

# TODO: Entrena un DecisionTreeClassifier(max_depth=4)
arbol = DecisionTreeClassifier(max_depth=4, random_state=42)
arbol.fit(X_train, y_train)
pred_arbol = arbol.predict(X_test)
# TODO: Imprime las matrices de confusión para ambos

print("Matriz de confusion k-NN (escalado):")
print(confusion_matrix(y_test, pred_knn))
print("\nMatriz de confusion k-NN (sin escalar):")
print(confusion_matrix(y_test, pred_knn_sin))
print("\nMatriz de confusion Arbol de Decision:")
print(confusion_matrix(y_test, pred_arbol))

## Evaluación Teórica / Decisión de Negocios

1. En este escenario médico, ¿qué implica un Error Tipo II (Falso Negativo) y por qué cuesta vidas?
2. ¿Por qué crees que el Árbol de Decisión no falló estrepitosamente incluso si no hubieses escalado los datos?

*Escribe tu respuesta aquí:*

**1.** Falso negativo (Error Tipo II): El modelo califica el tumor como benigno siendo maligno. La paciente no recibe tratamiento a tiempo y el cáncer avanza, lo que cuesta vidas. En medicina es prioritario reducirlos a cero, aun si aumentan los falsos positivos (que solo requieren pruebas adicionales).

**2.** Árbol de decisión sin escalado: No usa distancias (como k-NN, donde áreas grandes aplastan decimales), sino cortes individuales por variable (ej. "área > 700"). Como cambiar la escala no altera el orden de los datos, los puntos de corte y los resultados son exactamente iguales.